# CHSA Medical Triage Agent - Kaggle QLoRA

This notebook runs the project on a Kaggle GPU without reinstalling Kaggle's built-in PyTorch stack. It downloads the private Hugging Face dataset, audits it locally, then starts a small SFT smoke run.

Before running: enable a Kaggle GPU and add a Kaggle secret named `HF_TOKEN` with access to `Lokhidor/medical-triage-dataset`.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/Nhkp/medical-triage-agent.git"
REPO_DIR = Path("/kaggle/working/medical-triage-agent")

if not (REPO_DIR / ".git").exists():
    if REPO_DIR.exists():
        raise RuntimeError(
            f"{REPO_DIR} exists but is not a git checkout; remove it or choose another path"
        )
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print(Path.cwd())

## Install dependencies without reinstalling Torch

Kaggle already ships PyTorch. Installing a second `torch` wheel into `.venv` can exhaust disk space, so this notebook uses the system environment and installs only the missing project dependencies.

In [ ]:
!python -m pip install -q -U \
  datasets peft trl transformers accelerate bitsandbytes pyyaml trackio wrapt

In [ ]:
import torch

print("cuda_available=", torch.cuda.is_available())
print("device=", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
print("torch=", torch.__version__)

## Load Hugging Face credentials

The dataset is private. The token must be available as a Kaggle secret named `HF_TOKEN`.

In [ ]:
from kaggle_secrets import UserSecretsClient

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_DATASET_REPO"] = "Lokhidor/medical-triage-dataset"
os.environ.setdefault("HF_SFT_MODEL_REPO", "Lokhidor/chsa-qwen3-sft-lora")
os.environ.setdefault("HF_DPO_MODEL_REPO", "Lokhidor/chsa-qwen3-dpo-lora")

print("HF token loaded:", bool(os.environ.get("HF_TOKEN")))
print("Dataset repo:", os.environ["HF_DATASET_REPO"])

In [ ]:
!hf auth whoami

## Download the audited private dataset

The training configs expect local files under `data/processed/training`.

In [ ]:
!mkdir -p data/processed/training
!hf download "$HF_DATASET_REPO" \
  --type dataset \
  --local-dir data/processed/training \
  --include "sft_*.jsonl" \
  --include "dpo_*.jsonl" \
  --include "manifest.json" \
  --include "README.md"

In [ ]:
!PYTHONPATH=src python -m medical_triage_agent audit-training-data data/processed/training
!PYTHONPATH=src python -m medical_triage_agent summarize-training-data data/processed/training

## SFT smoke run

Start tiny: 5 steps, 32 samples. If this works, increase `--max-train-samples` before attempting the full dataset.

In [ ]:
!python scripts/train_sft.py \
  --config configs/sft_kaggle.yaml \
  --max-steps 5 \
  --max-train-samples 32

## Optional next steps

Run DPO only after `outputs/sft` exists. Push adapters to Hugging Face only after smoke runs are stable.

In [ ]:
# DPO smoke run, after SFT adapter exists
# !python scripts/train_dpo.py \
#   --config configs/dpo_kaggle.yaml \
#   --max-steps 5 \
#   --max-train-samples 32

In [ ]:
# Deterministic evaluation, after SFT adapter exists
# !python scripts/evaluate.py \
#   --config configs/sft_kaggle.yaml \
#   --model sft \
#   --adapter-path outputs/sft \
#   --output outputs/evaluations/sft.json